In [2]:
import torch
import math

In [34]:
batch_size = 2
seq_len = 9
n_heads = 2
d_model = 6
m = int(math.sqrt(seq_len))

k_neighbors = 2

In [35]:
k = torch.rand(batch_size, n_heads, m, m, d_model)
q = torch.rand(batch_size, n_heads, seq_len, d_model)

In [36]:
k.shape, q.shape

(torch.Size([2, 2, 3, 3, 6]), torch.Size([2, 2, 9, 6]))

In [37]:
k1 = k[..., : k.size(-1) // 2].sum(-2)
k2 = k[..., k.size(-1) // 2 :].sum(-3)

q1 = q[..., : q.size(-1) // 2]
q2 = q[..., q.size(-1) // 2 :]

In [38]:
k1.shape, k2.shape, q1.shape, q2.shape

(torch.Size([2, 2, 3, 3]),
 torch.Size([2, 2, 3, 3]),
 torch.Size([2, 2, 9, 3]),
 torch.Size([2, 2, 9, 3]))

In [39]:
# find which k1 are most similar to q1 without einsum
similarity1 = torch.matmul(q1, k1.transpose(-2, -1))
similarity2 = torch.matmul(q2, k2.transpose(-2, -1))

In [40]:
similarity1[0, 0, :, :]

tensor([[1.5555, 2.7242, 2.2865],
        [2.8626, 3.4660, 3.4761],
        [1.9590, 2.4441, 2.5735],
        [1.8486, 2.5304, 1.8580],
        [2.7049, 2.9972, 3.2799],
        [2.4663, 2.7399, 2.6197],
        [1.5179, 2.0295, 1.5603],
        [2.3880, 2.9251, 2.9101],
        [2.6786, 3.5399, 3.3065]])

In [41]:
values1, indices1 = torch.topk(similarity1, k=k_neighbors, dim=-1)

In [42]:
indices1.shape

torch.Size([2, 2, 9, 2])

In [43]:
ind_expanded1 = indices1.expand(-1, -1, -1, k1.size(-1))

RuntimeError: The expanded size of the tensor (3) must match the existing size (2) at non-singleton dimension 3.  Target sizes: [-1, -1, -1, 3].  Tensor sizes: [2, 2, 9, 2]